# Level 1 + 3 + 4: 
* Cài đặt cơ bản Eclat
* cài đặt tối ưu bằng cách sử dụng Bit Array và toán tử AND của Julia
* Đọc và xuất output SPMF

In [36]:
using Pkg
Pkg.activate("..")

# Nạp mã nguồn
include("../src/structures.jl")
include("../src/utils.jl")
include("../src/algorithm/eclat_optimized.jl")
include("../src/algorithm/eclat_basic.jl")

using .Structures, .Utils, .EclatOptimized, .EclatBasic
println("Đã nạp môi trường và cả hai phiên bản thuật toán Eclat.")

  Activating project at `d:\Github\ai biet\Lab2-DM`


Đã nạp môi trường và cả hai phiên bản thuật toán Eclat.


# --- LỰA CHỌN BENCHMARK ---
 1. "../data/toy/toy_data.txt"      (min_sup gợi ý: 3)
 2. "../data/benchmark/mushroom.txt" (min_sup gợi ý: 2000)
 3. "../data/benchmark/chess.txt"    (min_sup gợi ý: 2500)

In [37]:
data_path = "../data/benchmark/mushroom.txt" 
min_sup = 2000

2000

# --- LỰA CHỌN THUẬT TOÁN ---
* true  => Chạy bản Tối ưu (BitArray)
* false => Chạy bản Cơ bản (Set)


In [38]:
# --- LỰA CHỌN THUẬT TOÁN ---
# true  => Chạy bản Tối ưu (BitArray)
# false => Chạy bản Cơ bản (Set)
use_optimized = true 

true

In [39]:
println("File dữ liệu: ", data_path)
println("Ngưỡng min_sup: ", min_sup)
println("Thuật toán đang chọn: ", use_optimized ? "TỐI ƯU (BitArray)" : "CƠ BẢN (Set)")

# Đọc dữ liệu thô
tidsets, n_trans = Utils.read_spmf(data_path)

File dữ liệu: ../data/benchmark/mushroom.txt
Ngưỡng min_sup: 2000
Thuật toán đang chọn: TỐI ƯU (BitArray)


(Dict{Int64, BitVector}(5 => [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 56 => [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 55 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 114 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 110 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 123 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 60 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 30 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 32 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 6 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0]…), 8416)

In [40]:

results_optimized = Vector{ItemsetOptimized}()
results_basic = Vector{ItemsetBasic}()

if use_optimized
    # 1. Chuẩn bị dữ liệu cho bản Tối ưu
    P = [ItemsetOptimized([item], tidsets[item]) for item in sort(collect(keys(tidsets)))]
    filter!(x -> x.support >= min_sup, P)
    
    println("Đang chạy Eclat TỐI ƯU...")
    @time run_eclat_optimized(P, min_sup, results_optimized)
    
    final_results = results_optimized
else
    # 2. Chuẩn bị dữ liệu cho bản Cơ bản (Chuyển BitArray sang Set)
    P_basic = [ItemsetBasic([item], Set(findall(tidsets[item]))) for item in sort(collect(keys(tidsets)))]
    filter!(x -> x.support >= min_sup, P_basic)
    
    println("Đang chạy Eclat CƠ BẢN...")
    @time run_eclat_basic(P_basic, min_sup, results_basic)
    
    final_results = results_basic
end

println("\n HOÀN TẤT!")
println("Số lượng tập phổ biến tìm được: ", length(final_results))

UndefVarError: UndefVarError: `ItemsetOptimized` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.

In [41]:
# Tên file xuất ra
spmf_output_file = "output_spmf.txt"

# Gọi hàm write_results từ module Utils
Utils.write_results(final_results, spmf_output_file)

println("Đã xuất kết quả theo chuẩn SPMF ra file: $spmf_output_file")

Đã xuất kết quả theo chuẩn SPMF ra file: output_spmf.txt


In [42]:
println("Một số kết quả tìm được:")
println("-"^40)
for i in 1:min(10, length(final_results))
    r = final_results[i]
    println("Tập: $(r.items) | Support: $(r.support)")
end
println("-"^40)

Một số kết quả tìm được:
----------------------------------------
Tập: [1] | Support: 4488
Tập: [1, 5] | Support: 2084
Tập: [1, 5, 36] | Support: 2036
Tập: [1, 5, 36, 90] | Support: 2036
Tập: [1, 5, 36, 90, 94] | Support: 2036
Tập: [1, 5, 36, 94] | Support: 2036
Tập: [1, 5, 90] | Support: 2084
Tập: [1, 5, 90, 94] | Support: 2036
Tập: [1, 5, 94] | Support: 2036
Tập: [1, 23] | Support: 2752
----------------------------------------


# Phần 2: Tái tạo đúng kết quả

* Dùng lệnh này trong terminal:  julia --project=. tests/runtests.jl